In [1]:
import pandas as pd

# ==============================
# 1. Cargar CSV desde GitHub
# ==============================

url = "https://raw.githubusercontent.com/iJahir/etl-data-pipeline-25168282020-/refs/heads/main/data/raw/D_salarios.csv"

df = pd.read_csv(url)

print("Primeros registros:")
print(df.head())


# ==============================
# 2. Limpieza de datos
# ==============================

salarios = df.copy()

# limpiar nombres de columnas
salarios.columns = salarios.columns.str.strip().str.lower()

# limpiar espacios
for col in salarios.select_dtypes(include='object').columns:
    salarios[col] = salarios[col].astype(str).str.strip()

# eliminar duplicados
salarios = salarios.drop_duplicates()


# ==============================
# 3. Separar válidos y rechazados
# ==============================

validos = salarios[
    (~salarios.astype(str).apply(lambda x: x.str.contains("error|text")).any(axis=1)) &
    (salarios.notna().all(axis=1))
].copy()

rechazados = salarios.drop(validos.index).copy()


# ==============================
# 4. Motivo de rechazo
# ==============================

def motivo(row):
    motivos = []

    for col in salarios.columns:
        if pd.isna(row[col]):
            motivos.append(f"{col}_vacio")

        if "error" in str(row[col]):
            motivos.append(f"{col}_error")

        if "text" in str(row[col]):
            motivos.append(f"{col}_texto")

    return ",".join(motivos)


rechazados["motivo_rechazo"] = rechazados.apply(motivo, axis=1)


# ==============================
# 5. Exportar
# ==============================

validos.to_csv("curated_salarios.csv", index=False)
rechazados.to_csv("rejects_salarios.csv", index=False)

print("Salarios procesado correctamente ✅")

Primeros registros:
        id salario
0       32     NaN
1      NaN      35
2      NaN      35
3  text_88   error
4      NaN      35
Salarios procesado correctamente ✅


In [2]:
from google.colab import files

files.download('curated_salarios.csv')
files.download('rejects_salarios.csv')
#

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>